---
title: "Short Rate Models (Part 12: Local Momentum and Mean Reversion II)"
date: 2026-03-17
description: "We map Duan's full local-momentum term-structure model into the reduced package implementation, build the weekly rates panel with alphaforge, and fit the nonlinear model against a public weekly dataset."
categories: [Quantitative Finance, Interest Rate Models, Stochastic Processes]
content-type: "series"
series-id: "short-rate-models"
series-title: "Short Rate Models"
series-part: 12
project-ids: [short-rate-models, alphaforge]
---

## From Duan's Full Model to Our Reduced Package Model

The theory notebook derived the paper's formulas in full generality. The package implementation is intentionally narrower. The table below makes the mapping explicit.

| Paper block | Package status | Comment |
| --- | --- | --- |
| Local-momentum drift built from a recent-history summary | implemented exactly in spirit | The package keeps an explicit momentum state and a separate trend state. |
| Recursive local average / augmented Markov state | implemented exactly | The model state contains the recent-history summary rather than leaving it implicit. |
| Stochastic central tendency | approximated in reduced form | The package uses a single trend factor rather than the paper's full specification. |
| Local variation factor in the full term-structure system | omitted | The current implementation focuses on the nonlinear momentum channel. |
| Full no-arbitrage pricing-kernel estimation | omitted | The notebook uses reduced deterministic yield construction instead. |
| Weekly public yield panel | implemented exactly | `alphaforge` builds the weekly dataset from public daily rates resampled to a fixed Friday grid. |
| Full likelihood estimation | approximated in reduced form | The package fit uses regression-based calibration rather than the full MLE problem. |

The point of the notebook is therefore twofold. First, it demonstrates the public-data workflow. Second, it makes every approximation explicit rather than hiding it inside the code.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def locate_workspace() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'alphaforge').exists() and (candidate / 'short-rate-models').exists():
            return candidate
    raise RuntimeError('Could not locate the steveya workspace from the current working directory.')


WORKSPACE = locate_workspace()
sys.path.insert(0, str(WORKSPACE / 'alphaforge'))
sys.path.insert(0, str(WORKSPACE / 'short-rate-models'))

from alphaforge import (
    DataContext,
    DuckDBParquetStore,
    FREDDataSource,
    TradingCalendar,
    build_duan_weekly_dataset,
)
from short_rate_models import LocalMomentumTermStructureModel

In [ ]:
fred_api_key = os.environ.get('FRED_API_KEY')
if not fred_api_key:
    raise RuntimeError('Set FRED_API_KEY before running this notebook.')

ctx = DataContext(
    sources={'fred': FREDDataSource(api_key=fred_api_key)},
    calendars={'XNYS': TradingCalendar('XNYS', tz='UTC')},
    store=DuckDBParquetStore(root=WORKSPACE / '.alphaforge_store' / 'short_rate_models'),
)

dataset = build_duan_weekly_dataset(
    ctx,
    start=pd.Timestamp('1990-01-01', tz='UTC'),
    end=pd.Timestamp('2024-12-31', tz='UTC'),
)

dataset.short_rate.tail(), dataset.yields.tail()

## Reduced Fit

The reduced fit treats the observed weekly short rate as the local level and constructs latent trend and momentum summaries directly from the history of that short-rate proxy. This is much simpler than the full term-structure estimation problem in the paper, but it isolates the local-momentum mechanism transparently.

In [ ]:
model, fit = LocalMomentumTermStructureModel.fit(
    short_rate=dataset.short_rate,
    yields=dataset.yields,
)

fit['state_proxy'].tail()

In [ ]:
comparison = fit.get('yield_comparison')
if comparison is None:
    raise RuntimeError('The reduced fit did not produce a yield comparison block.')

errors = comparison['errors'].abs().mean().rename('mean_absolute_error')
errors

In [ ]:
path = pd.DataFrame(
    model.deterministic_path(steps=26, state=model.initial_state),
    columns=['trend', 'level', 'momentum'],
)
path.head()

## Interpretation

There are two questions to ask after the fit. First, does the estimated nonlinear momentum term matter numerically, or does the model collapse back to ordinary mean reversion? Second, when the nonlinear term does matter, does it improve the medium-horizon behavior of the fitted short rate and the fitted yield panel?

The reduced model earns its keep only if the answer to both questions is yes. If the nonlinear term is tiny, then the simpler affine Gaussian baseline is probably sufficient. If the term is large but unstable, then the paper's mechanism may be real but the reduced implementation may still be too crude for reliable empirical work.

## Limitations

This notebook is intentionally candid about what it does not do. It does not estimate the full pricing kernel, it does not recover every factor in the paper, and it does not claim exact no-arbitrage bond pricing. What it does do is fit the local nonlinear channel on a weekly public dataset, compare the resulting reduced-form yields to observed yields, and tie every implemented formula back to the derivation notebook.